In [1]:
import pandas as pd
from pathlib import Path

PROCESSED = Path("../data/processed")

# Read identifiers as plain text and keep strings such as "null" or "NA" unchanged
nodes = pd.read_csv(PROCESSED / "nodes.csv", dtype=str, keep_default_na=False)
edges = pd.read_csv(
    PROCESSED / "edges.csv",
    dtype={"source": str, "target": str},
    keep_default_na=False,
)

node_ids = set(nodes["deployment_id"])

source_missing = ~edges["source"].isin(node_ids)
target_missing = ~edges["target"].isin(node_ids)
dropped = edges[source_missing | target_missing]

missing_ids = pd.concat([
    edges.loc[source_missing, "source"],
    edges.loc[target_missing, "target"],
])

print("Nodes:", len(nodes))
print("Edges in edges.csv:", len(edges))
print("Edges dropped:", len(dropped))
print("Distinct missing identifiers:", missing_ids.nunique())
print(missing_ids.value_counts())

Nodes: 86489
Edges in edges.csv: 602295
Edges dropped: 1348
Distinct missing identifiers: 6
gauges.equilibria-xeqb.eth    634
equilibria-xeqb.eth           440
mimo.eth                      195
shardcoin.eth                  33
gaiadao.eth                    31
aventus.eth                    18
Name: count, dtype: int64


In [2]:
nodes_default = pd.read_csv(PROCESSED / "nodes.csv")
edges_default = pd.read_csv(PROCESSED / "edges.csv")

valid_default = set(nodes_default["deployment_id"])
kept_default = (
    edges_default["source"].isin(valid_default)
    & edges_default["target"].isin(valid_default)
)

print("Edges dropped with the default read (as in graph_builder.py):", (~kept_default).sum())

Edges dropped with the default read (as in graph_builder.py): 1348


In [7]:
both_missing = edges[source_missing & target_missing]
print("Edges with both endpoints missing:", len(both_missing))
print(both_missing.to_string(index=False))

Edges with both endpoints missing: 3
                    source                     target  shared_voters
       equilibria-xeqb.eth gauges.equilibria-xeqb.eth            212
       equilibria-xeqb.eth                   mimo.eth              1
gauges.equilibria-xeqb.eth                   mimo.eth              1


In [4]:
missing = sorted(missing_ids.unique())

deployments_raw = pd.read_csv(
    "../data/raw/deployments.csv", dtype=str, keep_default_na=False
)
raw_ids_lower = deployments_raw["deployment_id"].str.lower()
raw_names_lower = deployments_raw["name"].str.lower()

for dao in missing:
    exact = (deployments_raw["deployment_id"] == dao).sum()
    case_insensitive = (raw_ids_lower == dao).sum()

    # Core part of the identifier, e.g. "gauges.equilibria-xeqb.eth" -> "equilibria-xeqb"
    core = dao.removesuffix(".eth").split(".")[-1]
    similar = deployments_raw[
        raw_ids_lower.str.contains(core, regex=False)
        | raw_names_lower.str.contains(core, regex=False)
    ]

    print(f"\n{dao} | exact: {exact} | case-insensitive: {case_insensitive} | similar: {len(similar)}")
    print(similar[["deployment_id", "platform", "name", "unique_voters"]].head(10).to_string(index=False))


aventus.eth | exact: 0 | case-insensitive: 0 | similar: 1
  deployment_id platform            name unique_voters
aventus-gov.eth snapshot Aventus Network           0.0

equilibria-xeqb.eth | exact: 0 | case-insensitive: 0 | similar: 1
           deployment_id platform name unique_voters
eips.equilibria-xeqb.eth snapshot EIPs         230.0

gaiadao.eth | exact: 0 | case-insensitive: 0 | similar: 0
Empty DataFrame
Columns: [deployment_id, platform, name, unique_voters]
Index: []

gauges.equilibria-xeqb.eth | exact: 0 | case-insensitive: 0 | similar: 1
           deployment_id platform name unique_voters
eips.equilibria-xeqb.eth snapshot EIPs         230.0

mimo.eth | exact: 0 | case-insensitive: 0 | similar: 7
     deployment_id platform      name unique_voters
    mimokrypto.eth snapshot  Krypto73           2.0
     mimozirya.eth snapshot mimozirya           1.0
       emimoli.eth snapshot    winter           1.0
    mixazaruba.eth snapshot   Mimosaf           1.0
togoodtobetrue.eth sn

In [5]:
proposals = pd.read_csv(
    PROCESSED / "proposals_clean.csv", dtype=str, keep_default_na=False
)
proposals_missing = proposals[proposals["dao_id"].isin(missing)]

print("Proposals pointing to missing identifiers:", len(proposals_missing))
print(
    proposals_missing.groupby("dao_id").agg(
        platform=("platform", "first"),
        proposals=("proposal_id", "nunique"),
        first_proposal=("date", "min"),
        last_proposal=("date", "max"),
    )
)

Proposals pointing to missing identifiers: 319
                            platform  proposals       first_proposal  \
dao_id                                                                 
aventus.eth                 snapshot         22  2022-03-16 11:50:21   
equilibria-xeqb.eth         snapshot         50  2023-09-06 14:50:56   
gaiadao.eth                 snapshot          5  2021-05-03 01:54:18   
gauges.equilibria-xeqb.eth  snapshot         21  2023-10-09 08:44:53   
mimo.eth                    snapshot        212  2021-04-17 13:25:24   
shardcoin.eth               snapshot          9  2021-06-05 14:38:46   

                                  last_proposal  
dao_id                                           
aventus.eth                 2025-03-13 08:25:30  
equilibria-xeqb.eth         2024-08-05 10:40:46  
gaiadao.eth                 2021-06-10 17:06:49  
gauges.equilibria-xeqb.eth  2024-02-26 15:51:55  
mimo.eth                    2025-05-16 12:44:51  
shardcoin.eth             

In [6]:
chunks = []
for chunk in pd.read_csv(
    PROCESSED / "votes_clean.csv",
    usecols=["proposal_id", "deployment_id", "platform", "date", "voter"],
    dtype=str,
    keep_default_na=False,
    chunksize=1_000_000,
):
    chunks.append(chunk[chunk["deployment_id"].isin(missing)])

votes_missing = pd.concat(chunks, ignore_index=True)

print(
    votes_missing.groupby("deployment_id").agg(
        platform=("platform", "first"),
        votes=("voter", "size"),
        voters=("voter", "nunique"),
        proposals=("proposal_id", "nunique"),
        first_vote=("date", "min"),
        last_vote=("date", "max"),
    )
)

                            platform  votes  voters  proposals  \
deployment_id                                                    
aventus.eth                 snapshot    233      76         22   
equilibria-xeqb.eth         snapshot   1949     290         50   
gaiadao.eth                 snapshot    120      69          5   
gauges.equilibria-xeqb.eth  snapshot   4968     647         21   
mimo.eth                    snapshot   1792     104        212   
shardcoin.eth               snapshot    229     110          9   

                                     first_vote            last_vote  
deployment_id                                                         
aventus.eth                 2022-03-16 12:08:40  2025-03-14 21:26:02  
equilibria-xeqb.eth         2023-09-06 14:53:11  2024-08-11 17:53:22  
gaiadao.eth                 2021-05-03 02:18:36  2021-06-14 18:31:33  
gauges.equilibria-xeqb.eth  2023-10-09 08:59:16  2024-02-28 05:59:11  
mimo.eth                    2021-04-22 08:56:

## Impact of including the missing deployments

Output of `python src/graph_sanity.py` before and after applying the S2-02 policy.

| Metric | Before | After | Change |
| :-- | --: | --: | --: |
| Nodes | 86,489 | 86,495 | +6 |
| Edges | 600,947 | 602,295 | +1,348 |
| Isolated nodes | 49,243 | 49,243 | 0 |
| Connected components | 50,754 | 50,754 | 0 |
| Largest connected component | 31,024 | 31,030 | +6 |
| Average degree | 13.90 | 13.93 | +0.03 |
| Maximum degree | 10,237 | 10,240 | +3 |

All six deployments join the largest connected component. No edge is dropped any more.